In [1]:
import importlib.util
import subprocess
import sys

missing_packages = [package for package in ["optuna", "psutil"] if importlib.util.find_spec(package) is None]

if missing_packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing_packages])
    print("Packages installed. Restart the kernel once, then run the notebook again.")
else:
    print("Required packages are already installed.")

print("Python executable:", sys.executable)

Required packages are already installed.
Python executable: c:\Users\USER\miniconda3\envs\ml_project\python.exe


In [2]:
import os

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"

import gc
import json
import shutil
import sys
import time
import warnings
import zipfile
import py_compile

from pathlib import Path

import joblib
import numpy as np
import optuna
import pandas as pd
import psutil
import sklearn

from joblib import Parallel, delayed
from scipy import sparse
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.metrics import accuracy_score, log_loss, roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold
from threadpoolctl import threadpool_limits

warnings.filterwarnings("ignore", category=ConvergenceWarning)
optuna.logging.set_verbosity(optuna.logging.WARNING)

RANDOM_STATE = 42
physical_cores = psutil.cpu_count(logical=False) or 4
logical_cores = psutil.cpu_count(logical=True) or physical_cores
total_ram_gb = psutil.virtual_memory().total / 1024**3
available_ram_gb = psutil.virtual_memory().available / 1024**3
N_WORKERS = min(8, max(2, physical_cores))
CV_WORKERS = min(3, N_WORKERS)

print("Python          :", sys.version.split()[0])
print("Python path     :", sys.executable)
print("Scikit-learn   :", sklearn.__version__)
print("Optuna         :", optuna.__version__)
print("Physical cores :", physical_cores)
print("Logical cores  :", logical_cores)
print("Total RAM      :", f"{total_ram_gb:.1f} GB")
print("Available RAM  :", f"{available_ram_gb:.1f} GB")
print("Optuna workers :", N_WORKERS)
print("CV workers     :", CV_WORKERS)

Python          : 3.10.20
Python path     : c:\Users\USER\miniconda3\envs\ml_project\python.exe
Scikit-learn   : 1.7.2
Optuna         : 4.9.0
Physical cores : 8
Logical cores  : 16
Total RAM      : 31.6 GB
Available RAM  : 17.1 GB
Optuna workers : 8
CV workers     : 3


In [3]:
current_dir = Path.cwd().resolve()
project_root = next((path for path in [current_dir, *current_dir.parents] if (path / "Trace-The-Race-Dataset").is_dir()), None)

if project_root is None:
    raise FileNotFoundError("Trace-The-Race-Dataset folder was not found.")

dataset_root = project_root / "Trace-The-Race-Dataset"
feature_dir = dataset_root / "outputs" / "05_feature_engineering"
model_dir = dataset_root / "outputs" / "06_model_training"
optuna_dir = model_dir / "optuna_baseline_v3"
submission_dir = model_dir / "submissions"

optuna_dir.mkdir(parents=True, exist_ok=True)
submission_dir.mkdir(parents=True, exist_ok=True)

print("Project root :", project_root)
print("Feature dir  :", feature_dir)
print("Optuna dir   :", optuna_dir)

Project root : C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Trace-The-Race-Competition
Feature dir  : C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Trace-The-Race-Competition\Trace-The-Race-Dataset\outputs\05_feature_engineering
Optuna dir   : C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Trace-The-Race-Competition\Trace-The-Race-Dataset\outputs\06_model_training\optuna_baseline_v3


In [4]:
train_metadata = pd.read_parquet(feature_dir / "train_metadata.parquet")
validation_metadata = pd.read_parquet(feature_dir / "validation_metadata.parquet")

y_train = np.load(feature_dir / "train_labels.npy")
y_validation = np.load(feature_dir / "validation_labels.npy")

X_train_word = sparse.load_npz(feature_dir / "train_word_tfidf.npz").tocsr()
X_validation_word = sparse.load_npz(feature_dir / "validation_word_tfidf.npz").tocsr()

X_train_char = sparse.load_npz(feature_dir / "train_char_tfidf.npz").tocsr()
X_validation_char = sparse.load_npz(feature_dir / "validation_char_tfidf.npz").tocsr()

X_train_word_char = sparse.hstack([X_train_word, X_train_char], format="csr")
X_validation_word_char = sparse.hstack([X_validation_word, X_validation_char], format="csr")

groups_train = train_metadata["session_id"].astype("string").fillna("").to_numpy()
groups_validation = validation_metadata["session_id"].astype("string").fillna("").to_numpy()

print("Train rows          :", f"{len(y_train):,}")
print("Validation rows     :", f"{len(y_validation):,}")
print("Train sessions      :", f"{pd.Series(groups_train).nunique():,}")
print("Validation sessions :", f"{pd.Series(groups_validation).nunique():,}")
print("Train positive rate :", f"{y_train.mean():.6f}")
print("Validation rate     :", f"{y_validation.mean():.6f}")
print("Word matrix         :", X_train_word.shape)
print("Char matrix         :", X_train_char.shape)
print("Word + char matrix  :", X_train_word_char.shape)

Train rows          : 28,125
Validation rows     : 6,947
Train sessions      : 18,256
Validation sessions : 4,565
Train positive rate : 0.699484
Validation rate     : 0.714553
Word matrix         : (28125, 60000)
Char matrix         : (28125, 60000)
Word + char matrix  : (28125, 120000)


In [5]:
session_overlap = set(groups_train) & set(groups_validation)

if session_overlap:
    raise ValueError("Session leakage found between train and validation.")

internal_splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
internal_train_idx, internal_valid_idx = next(internal_splitter.split(np.zeros(len(y_train)), y_train, groups_train))

y_internal_train = y_train[internal_train_idx]
y_internal_valid = y_train[internal_valid_idx]

X_internal_train_word = X_train_word[internal_train_idx]
X_internal_valid_word = X_train_word[internal_valid_idx]
X_internal_train_char = X_train_char[internal_train_idx]
X_internal_valid_char = X_train_char[internal_valid_idx]
X_internal_train_word_char = X_train_word_char[internal_train_idx]
X_internal_valid_word_char = X_train_word_char[internal_valid_idx]

print("Train-validation session overlap:", len(session_overlap))
print("Internal train rows              :", f"{len(internal_train_idx):,}")
print("Internal validation rows         :", f"{len(internal_valid_idx):,}")
print("Internal session overlap         :", len(set(groups_train[internal_train_idx]) & set(groups_train[internal_valid_idx])))

Train-validation session overlap: 0
Internal train rows              : 22,451
Internal validation rows         : 5,674
Internal session overlap         : 0


In [6]:
def get_internal_matrices(matrix_key):
    if matrix_key == "word":
        return X_internal_train_word, X_internal_valid_word
    if matrix_key == "char":
        return X_internal_train_char, X_internal_valid_char
    if matrix_key == "word_char":
        return X_internal_train_word_char, X_internal_valid_word_char
    raise ValueError(f"Unknown matrix key: {matrix_key}")


def get_train_matrix(matrix_key):
    if matrix_key == "word":
        return X_train_word
    if matrix_key == "char":
        return X_train_char
    if matrix_key == "word_char":
        return X_train_word_char
    raise ValueError(f"Unknown matrix key: {matrix_key}")


def get_validation_matrix(matrix_key):
    if matrix_key == "word":
        return X_validation_word
    if matrix_key == "char":
        return X_validation_char
    if matrix_key == "word_char":
        return X_validation_word_char
    raise ValueError(f"Unknown matrix key: {matrix_key}")


def get_matrix_key(model_key):
    if model_key.startswith("word_char"):
        return "word_char"
    if model_key.startswith("word"):
        return "word"
    if model_key.startswith("char"):
        return "char"
    raise ValueError(f"Unknown model key: {model_key}")


def get_model_family(model_key):
    if model_key.endswith("logistic"):
        return "logistic"
    if model_key.endswith("sgd"):
        return "sgd"
    raise ValueError(f"Unknown model family: {model_key}")

In [7]:
def build_model(params, final_fit=False):
    model_key = params["model_key"]
    model_family = get_model_family(model_key)

    if model_family == "logistic":
        return LogisticRegression(C=float(params["log_C"]), solver=params["log_solver"], penalty="l2", tol=float(params["log_tol"]), fit_intercept=bool(params["log_fit_intercept"]), class_weight=params["log_class_weight"], max_iter=800 if final_fit else 350, random_state=RANDOM_STATE)

    average_value = False if params["sgd_average"] == "false" else int(params["sgd_average"])

    return SGDClassifier(loss="log_loss", penalty=params["sgd_penalty"], alpha=float(params["sgd_alpha"]), l1_ratio=float(params.get("sgd_l1_ratio", 0.15)), tol=float(params["sgd_tol"]), fit_intercept=bool(params["sgd_fit_intercept"]), class_weight=params["sgd_class_weight"], average=average_value, max_iter=3000 if final_fit else 1800, shuffle=True, random_state=RANDOM_STATE)

In [8]:
MODEL_CHOICES = ["word_logistic", "word_char_logistic", "word_sgd", "char_sgd", "word_char_sgd"]

def objective(trial):
    model_key = trial.suggest_categorical("model_key", MODEL_CHOICES)
    model_family = get_model_family(model_key)
    matrix_key = get_matrix_key(model_key)
    params = {"model_key": model_key}

    if model_family == "logistic":
        params["log_C"] = trial.suggest_float("log_C", 1e-3, 20.0, log=True)
        params["log_solver"] = "saga" if matrix_key == "word_char" else trial.suggest_categorical("log_solver", ["liblinear", "saga"])
        params["log_tol"] = trial.suggest_float("log_tol", 1e-5, 3e-3, log=True)
        params["log_fit_intercept"] = trial.suggest_categorical("log_fit_intercept", [True, False])
        params["log_class_weight"] = trial.suggest_categorical("log_class_weight", [None, "balanced"])

    if model_family == "sgd":
        params["sgd_alpha"] = trial.suggest_float("sgd_alpha", 1e-7, 1e-2, log=True)
        params["sgd_penalty"] = trial.suggest_categorical("sgd_penalty", ["l2", "elasticnet"])
        params["sgd_tol"] = trial.suggest_float("sgd_tol", 1e-6, 2e-3, log=True)
        params["sgd_fit_intercept"] = trial.suggest_categorical("sgd_fit_intercept", [True, False])
        params["sgd_class_weight"] = trial.suggest_categorical("sgd_class_weight", [None, "balanced"])
        params["sgd_average"] = trial.suggest_categorical("sgd_average", ["false", "50", "100"])

        if params["sgd_penalty"] == "elasticnet":
            params["sgd_l1_ratio"] = trial.suggest_float("sgd_l1_ratio", 0.02, 0.80)

    X_fit, X_eval = get_internal_matrices(matrix_key)
    model = build_model(params, final_fit=False)
    start_time = time.perf_counter()

    with warnings.catch_warnings(), threadpool_limits(limits=1):
        warnings.simplefilter("ignore")
        model.fit(X_fit, y_internal_train)

    probabilities = np.clip(model.predict_proba(X_eval)[:, 1], 1e-6, 1 - 1e-6)
    score = log_loss(y_internal_valid, probabilities)

    trial.set_user_attr("matrix_key", matrix_key)
    trial.set_user_attr("model_family", model_family)
    trial.set_user_attr("fit_seconds", round(time.perf_counter() - start_time, 3))
    trial.set_user_attr("probability_mean", float(probabilities.mean()))
    trial.set_user_attr("probability_min", float(probabilities.min()))
    trial.set_user_attr("probability_max", float(probabilities.max()))

    del model, probabilities
    gc.collect()

    return score

In [9]:
optuna_database = optuna_dir / "optuna_tuning.db"
storage_url = f"sqlite:///{optuna_database.as_posix()}"

sampler = optuna.samplers.TPESampler(seed=RANDOM_STATE, n_startup_trials=18)
study = optuna.create_study(study_name="combined_sparse_baseline_v3", storage=storage_url, load_if_exists=True, direction="minimize", sampler=sampler)

completed_trials = sum(trial.state == optuna.trial.TrialState.COMPLETE for trial in study.trials)
failed_trials = sum(trial.state == optuna.trial.TrialState.FAIL for trial in study.trials)

print("Completed trials :", completed_trials)
print("Failed trials    :", failed_trials)
print("Best score       :", study.best_value if completed_trials else "Not available")
print("Database         :", optuna_database)

Completed trials : 0
Failed trials    : 0
Best score       : Not available
Database         : C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Trace-The-Race-Competition\Trace-The-Race-Dataset\outputs\06_model_training\optuna_baseline_v3\optuna_tuning.db


In [10]:
TARGET_TRIALS = 72
TIMEOUT_MINUTES = 90

completed_trials = sum(trial.state == optuna.trial.TrialState.COMPLETE for trial in study.trials)
remaining_trials = max(0, TARGET_TRIALS - completed_trials)

print("Target trials    :", TARGET_TRIALS)
print("Completed trials :", completed_trials)
print("Remaining trials :", remaining_trials)
print("Parallel workers :", N_WORKERS)
print("Timeout minutes  :", TIMEOUT_MINUTES)

if remaining_trials > 0:
    study.optimize(objective, n_trials=remaining_trials, timeout=TIMEOUT_MINUTES * 60, n_jobs=N_WORKERS, gc_after_trial=True, show_progress_bar=True)

print("Tuning completed.")
print("Best internal Log Loss:", f"{study.best_value:.6f}")
print(json.dumps(study.best_params, indent=2))

Target trials    : 72
Completed trials : 0
Remaining trials : 72
Parallel workers : 8
Timeout minutes  : 90


  0%|          | 0/72 [00:00<?, ?it/s]

Tuning completed.
Best internal Log Loss: 0.570895
{
  "model_key": "word_char_logistic",
  "log_C": 0.49004624688437864,
  "log_tol": 5.7769617799635894e-05,
  "log_fit_intercept": true,
  "log_class_weight": null
}


In [11]:
trials_df = study.trials_dataframe()
trials_df.to_csv(optuna_dir / "all_optuna_trials.csv", index=False)

completed_trial_objects = sorted([trial for trial in study.trials if trial.state == optuna.trial.TrialState.COMPLETE], key=lambda trial: trial.value)

TOP_CONFIGS = 5
top_candidates = []
seen_signatures = set()

for trial in completed_trial_objects:
    signature = json.dumps(trial.params, sort_keys=True, default=str)

    if signature in seen_signatures:
        continue

    seen_signatures.add(signature)
    top_candidates.append({"trial_number": trial.number, "internal_log_loss": float(trial.value), "params": trial.params})

    if len(top_candidates) >= TOP_CONFIGS:
        break

print("Selected top configurations:", len(top_candidates))

for rank, candidate in enumerate(top_candidates, start=1):
    print(rank, candidate["trial_number"], candidate["params"]["model_key"], f"{candidate['internal_log_loss']:.6f}")

Selected top configurations: 5
1 59 word_char_logistic 0.570895
2 43 word_char_logistic 0.570897
3 11 word_char_logistic 0.570906
4 58 word_char_logistic 0.571152
5 44 word_char_logistic 0.571185


In [13]:
def complete_params(params):
    params = dict(params)
    model_key = params["model_key"]
    model_family = get_model_family(model_key)
    matrix_key = get_matrix_key(model_key)

    if model_family == "logistic":
        params.setdefault("log_C", 1.0)
        params.setdefault("log_solver", "saga" if matrix_key == "word_char" else "liblinear")
        params.setdefault("log_tol", 1e-4)
        params.setdefault("log_fit_intercept", True)
        params.setdefault("log_class_weight", None)

    if model_family == "sgd":
        params.setdefault("sgd_alpha", 1e-4)
        params.setdefault("sgd_penalty", "l2")
        params.setdefault("sgd_tol", 1e-3)
        params.setdefault("sgd_fit_intercept", True)
        params.setdefault("sgd_class_weight", None)
        params.setdefault("sgd_average", "false")
        params.setdefault("sgd_l1_ratio", 0.15)

    return params


def build_model(params, final_fit=False):
    params = complete_params(params)
    model_key = params["model_key"]
    model_family = get_model_family(model_key)

    if model_family == "logistic":
        return LogisticRegression(C=float(params["log_C"]), solver=params["log_solver"], penalty="l2", tol=float(params["log_tol"]), fit_intercept=bool(params["log_fit_intercept"]), class_weight=params["log_class_weight"], max_iter=800 if final_fit else 350, random_state=RANDOM_STATE)

    average_value = False if params["sgd_average"] == "false" else int(params["sgd_average"])

    return SGDClassifier(loss="log_loss", penalty=params["sgd_penalty"], alpha=float(params["sgd_alpha"]), l1_ratio=float(params["sgd_l1_ratio"]), tol=float(params["sgd_tol"]), fit_intercept=bool(params["sgd_fit_intercept"]), class_weight=params["sgd_class_weight"], average=average_value, max_iter=3000 if final_fit else 1800, shuffle=True, random_state=RANDOM_STATE)

In [14]:
cv_splitter = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
cv_splits = list(cv_splitter.split(np.zeros(len(y_train)), y_train, groups_train))

def fit_one_fold(params, fit_idx, valid_idx):
    matrix_key = get_matrix_key(params["model_key"])
    X_matrix = get_train_matrix(matrix_key)
    model = build_model(params, final_fit=True)
    start_time = time.perf_counter()

    with warnings.catch_warnings(), threadpool_limits(limits=1):
        warnings.simplefilter("ignore")
        model.fit(X_matrix[fit_idx], y_train[fit_idx])

    probabilities = np.clip(model.predict_proba(X_matrix[valid_idx])[:, 1], 1e-6, 1 - 1e-6)
    fold_loss = log_loss(y_train[valid_idx], probabilities)
    elapsed_seconds = time.perf_counter() - start_time

    return valid_idx, probabilities, fold_loss, elapsed_seconds

cv_candidate_results = []
candidate_oof_predictions = {}

for candidate in top_candidates:
    params = candidate["params"]
    model_key = params["model_key"]

    print("Evaluating:", model_key, "trial", candidate["trial_number"])

    fold_outputs = Parallel(n_jobs=CV_WORKERS, prefer="threads")(delayed(fit_one_fold)(params, fit_idx, valid_idx) for fit_idx, valid_idx in cv_splits)

    oof_probabilities = np.full(len(y_train), np.nan, dtype=np.float64)
    fold_losses = []
    fold_times = []

    for valid_idx, probabilities, fold_loss, elapsed_seconds in fold_outputs:
        oof_probabilities[valid_idx] = probabilities
        fold_losses.append(float(fold_loss))
        fold_times.append(float(elapsed_seconds))

    if np.isnan(oof_probabilities).any():
        raise ValueError(f"Missing OOF predictions for {model_key}")

    oof_loss = log_loss(y_train, oof_probabilities)
    oof_auc = roc_auc_score(y_train, oof_probabilities)
    candidate_name = f"trial_{candidate['trial_number']}_{model_key}"
    candidate_oof_predictions[candidate_name] = oof_probabilities

    cv_candidate_results.append({"candidate_name": candidate_name, "trial_number": candidate["trial_number"], "model_key": model_key, "matrix_key": get_matrix_key(model_key), "model_family": get_model_family(model_key), "internal_log_loss": candidate["internal_log_loss"], "cv_log_loss": float(oof_loss), "cv_auc": float(oof_auc), "fold_1_loss": fold_losses[0], "fold_2_loss": fold_losses[1], "fold_3_loss": fold_losses[2], "cv_std": float(np.std(fold_losses)), "training_seconds": float(sum(fold_times)), "params": params})

    print("OOF Log Loss:", f"{oof_loss:.6f}")
    print("Fold losses :", [round(value, 6) for value in fold_losses])

cv_results_df = pd.DataFrame([{key: value for key, value in result.items() if key != "params"} for result in cv_candidate_results])
cv_results_df = cv_results_df.sort_values(["cv_log_loss", "cv_std"]).reset_index(drop=True)
cv_results_df.to_csv(optuna_dir / "top_candidate_cv_results.csv", index=False)

display(cv_results_df)

Evaluating: word_char_logistic trial 59
OOF Log Loss: 0.562196
Fold losses : [0.557764, 0.560578, 0.568268]
Evaluating: word_char_logistic trial 43
OOF Log Loss: 0.562177
Fold losses : [0.557748, 0.560542, 0.568261]
Evaluating: word_char_logistic trial 11
OOF Log Loss: 0.562109
Fold losses : [0.5577, 0.560409, 0.568239]
Evaluating: word_char_logistic trial 58
OOF Log Loss: 0.562804
Fold losses : [0.558291, 0.56146, 0.568684]
Evaluating: word_char_logistic trial 44
OOF Log Loss: 0.562862
Fold losses : [0.558344, 0.561537, 0.568728]


,candidate_name,trial_number,model_key,matrix_key,model_family,internal_log_loss,cv_log_loss,cv_auc,fold_1_loss,fold_2_loss,fold_3_loss,cv_std,training_seconds
0,trial_11_word_char_logistic,11,word_char_logistic,word_char,logistic,0.570906,0.562109,0.692319,0.557700,0.560409,0.568239,0.004469,50.044694
1,trial_43_word_char_logistic,43,word_char_logistic,word_char,logistic,0.570897,0.562177,0.692411,0.557748,0.560542,0.568261,0.004446,44.299177
2,trial_59_word_char_logistic,59,word_char_logistic,word_char,logistic,0.570895,0.562196,0.692436,0.557764,0.560578,0.568268,0.004440,108.864556
3,trial_58_word_char_logistic,58,word_char_logistic,word_char,logistic,0.571152,0.562804,0.692522,0.558291,0.561460,0.568684,0.004349,47.075924
4,trial_44_word_char_logistic,44,word_char_logistic,word_char,logistic,0.571185,0.562862,0.692511,0.558344,0.561537,0.568728,0.004343,45.387402


In [15]:
selected_cv_result = cv_results_df.iloc[0]
selected_candidate_name = selected_cv_result["candidate_name"]
selected_trial_number = int(selected_cv_result["trial_number"])
selected_candidate = next(result for result in cv_candidate_results if result["trial_number"] == selected_trial_number)

selected_params = selected_candidate["params"]
selected_model_key = selected_candidate["model_key"]
selected_matrix_key = selected_candidate["matrix_key"]
selected_model_family = selected_candidate["model_family"]
selected_oof_probabilities = candidate_oof_predictions[selected_candidate_name]

oof_prior = float(y_train.mean())
alpha_values = np.linspace(0.20, 1.00, 161)
alpha_scores = [log_loss(y_train, alpha * selected_oof_probabilities + (1 - alpha) * oof_prior) for alpha in alpha_values]

best_alpha_index = int(np.argmin(alpha_scores))
best_alpha = float(alpha_values[best_alpha_index])
raw_oof_loss = log_loss(y_train, selected_oof_probabilities)
calibrated_oof_loss = float(alpha_scores[best_alpha_index])

print("Selected trial     :", selected_trial_number)
print("Selected model     :", selected_model_key)
print("Selected matrix    :", selected_matrix_key)
print("Three-fold OOF Loss:", f"{selected_candidate['cv_log_loss']:.6f}")
print("Raw OOF Log Loss   :", f"{raw_oof_loss:.6f}")
print("Best alpha         :", f"{best_alpha:.4f}")
print("Calibrated OOF Loss:", f"{calibrated_oof_loss:.6f}")
print(json.dumps(selected_params, indent=2))

Selected trial     : 11
Selected model     : word_char_logistic
Selected matrix    : word_char
Three-fold OOF Loss: 0.562109
Raw OOF Log Loss   : 0.562109
Best alpha         : 1.0000
Calibrated OOF Loss: 0.562109
{
  "model_key": "word_char_logistic",
  "log_C": 0.5255234497612964,
  "log_tol": 0.0001080111513918553,
  "log_fit_intercept": true,
  "log_class_weight": null
}


In [16]:
X_train_selected = get_train_matrix(selected_matrix_key)
X_validation_selected = get_validation_matrix(selected_matrix_key)
validation_model = build_model(selected_params, final_fit=True)

with warnings.catch_warnings(), threadpool_limits(limits=1):
    warnings.simplefilter("ignore")
    validation_model.fit(X_train_selected, y_train)

validation_raw_probabilities = np.clip(validation_model.predict_proba(X_validation_selected)[:, 1], 1e-6, 1 - 1e-6)
validation_prior = float(y_train.mean())
validation_calibrated_probabilities = np.clip(best_alpha * validation_raw_probabilities + (1 - best_alpha) * validation_prior, 1e-6, 1 - 1e-6)

raw_validation_loss = log_loss(y_validation, validation_raw_probabilities)
calibrated_validation_loss = log_loss(y_validation, validation_calibrated_probabilities)
raw_validation_auc = roc_auc_score(y_validation, validation_raw_probabilities)
raw_validation_accuracy = accuracy_score(y_validation, validation_raw_probabilities >= 0.5)

use_calibration = calibrated_validation_loss <= raw_validation_loss
final_alpha = best_alpha if use_calibration else 1.0

np.save(optuna_dir / "selected_oof_probabilities.npy", selected_oof_probabilities)
np.save(optuna_dir / "validation_raw_probabilities.npy", validation_raw_probabilities)
np.save(optuna_dir / "validation_calibrated_probabilities.npy", validation_calibrated_probabilities)

selection_report = {"selected_trial_number": selected_trial_number, "selected_model_key": selected_model_key, "selected_matrix_key": selected_matrix_key, "selected_model_family": selected_model_family, "selected_params": selected_params, "internal_log_loss": float(selected_candidate["internal_log_loss"]), "three_fold_oof_log_loss": float(selected_candidate["cv_log_loss"]), "three_fold_oof_auc": float(selected_candidate["cv_auc"]), "three_fold_cv_std": float(selected_candidate["cv_std"]), "calibration_alpha_from_oof": best_alpha, "calibration_used": use_calibration, "final_alpha": final_alpha, "raw_validation_log_loss": float(raw_validation_loss), "calibrated_validation_log_loss": float(calibrated_validation_loss), "validation_auc": float(raw_validation_auc), "validation_accuracy": float(raw_validation_accuracy), "sklearn_version": sklearn.__version__, "optuna_version": optuna.__version__, "optuna_workers": N_WORKERS}

with open(optuna_dir / "selected_model_report.json", "w", encoding="utf-8") as file:
    json.dump(selection_report, file, indent=2)

print("Raw validation Log Loss       :", f"{raw_validation_loss:.6f}")
print("Calibrated validation Log Loss:", f"{calibrated_validation_loss:.6f}")
print("Validation AUROC              :", f"{raw_validation_auc:.6f}")
print("Calibration used              :", use_calibration)
print("Final alpha                   :", f"{final_alpha:.4f}")

Raw validation Log Loss       : 0.548239
Calibrated validation Log Loss: 0.548239
Validation AUROC              : 0.698239
Calibration used              : True
Final alpha                   : 1.0000


In [17]:
if selected_matrix_key == "word":
    X_full = sparse.vstack([X_train_word, X_validation_word], format="csr")
elif selected_matrix_key == "char":
    X_full = sparse.vstack([X_train_char, X_validation_char], format="csr")
elif selected_matrix_key == "word_char":
    X_full = sparse.vstack([X_train_word_char, X_validation_word_char], format="csr")
else:
    raise ValueError(f"Unknown selected matrix: {selected_matrix_key}")

y_full = np.concatenate([y_train, y_validation])
final_prior = float(y_full.mean())
final_model = build_model(selected_params, final_fit=True)

with warnings.catch_warnings(), threadpool_limits(limits=1):
    warnings.simplefilter("ignore")
    final_model.fit(X_full, y_full)

joblib.dump(final_model, optuna_dir / "tuned_final_model_local_backup.joblib")

if list(final_model.classes_) != [0, 1]:
    raise ValueError(f"Unexpected class order: {final_model.classes_}")

model_coef = np.asarray(final_model.coef_, dtype=np.float32).reshape(-1)
model_intercept = float(np.asarray(final_model.intercept_).reshape(-1)[0])

np.savez_compressed(optuna_dir / "linear_model_weights.npz", coef=model_coef, intercept=np.float32(model_intercept))

print("Final model trained.")
print("Full matrix       :", X_full.shape)
print("Coefficient count :", len(model_coef))
print("Intercept         :", model_intercept)

Final model trained.
Full matrix       : (35072, 120000)
Coefficient count : 120000
Intercept         : 0.4478737711906433


In [18]:
runtime_code = 'from pathlib import Path\nimport json\n\nimport joblib\nimport numpy as np\nimport pandas as pd\nfrom scipy import sparse\n\n\nROOT = Path(__file__).resolve().parent\nDATA_DIR = ROOT / "data"\nASSET_DIR = ROOT / "assets"\n\n\ndef find_column(columns, candidates):\n    for candidate in candidates:\n        if candidate in columns:\n            return candidate\n    for column in columns:\n        for candidate in candidates:\n            if candidate in column:\n                return column\n    return None\n\n\ndef normalize_role(value):\n    value = str(value).strip().lower()\n    if any(word in value for word in ["student", "learner", "user", "pupil"]):\n        return "student"\n    if any(word in value for word in ["tutor", "teacher", "assistant", "instructor"]):\n        return "tutor"\n    return "background"\n\n\ndef find_transcript_file(session_id):\n    transcript_dir = DATA_DIR / "test_transcripts"\n    exact_file = transcript_dir / f"{session_id}.csv"\n    if exact_file.exists():\n        return exact_file\n    matched_files = list(transcript_dir.glob(f"{session_id}.*"))\n    if not matched_files:\n        raise FileNotFoundError(f"Transcript not found for session: {session_id}")\n    return matched_files[0]\n\n\ndef extract_session_text(session_id):\n    transcript_file = find_transcript_file(session_id)\n    transcript = pd.read_csv(transcript_file)\n    transcript.columns = transcript.columns.astype(str).str.replace("\\ufeff", "", regex=False).str.strip().str.lower()\n    role_column = find_column(transcript.columns, ["role", "speaker", "participant", "author"])\n    content_column = find_column(transcript.columns, ["content", "text", "message", "utterance", "transcript"])\n    if role_column is None or content_column is None:\n        raise ValueError(f"Role/content columns were not found in {transcript_file.name}")\n    roles = transcript[role_column].map(normalize_role)\n    contents = transcript[content_column].fillna("").astype(str).str.replace(r"\\s+", " ", regex=True).str.strip()\n    transcript_text = "\\n".join(f"[{role.upper()}] {content}" for role, content in zip(roles, contents) if content)\n    return {"session_id": str(session_id), "transcript_text": transcript_text}\n\n\ndef stable_sigmoid(values):\n    values = np.clip(values, -35.0, 35.0)\n    return 1.0 / (1.0 + np.exp(-values))\n\n\ndef main():\n    model_info = json.loads((ASSET_DIR / "model_info.json").read_text(encoding="utf-8"))\n    model_weights = np.load(ASSET_DIR / "linear_model_weights.npz")\n    coef = model_weights["coef"].astype(np.float32)\n    intercept = float(model_weights["intercept"])\n\n    test_features = pd.read_csv(DATA_DIR / "test_features.csv")\n    test_features.columns = test_features.columns.astype(str).str.replace("\\ufeff", "", regex=False).str.strip().str.lower()\n    test_features["response_id"] = test_features["response_id"].astype("string").str.strip()\n    test_features["session_id"] = test_features["session_id"].astype("string").str.strip()\n    test_features["learning_objective"] = test_features["learning_objective"].fillna("").astype("string").str.replace(r"\\s+", " ", regex=True).str.strip()\n\n    session_text = pd.DataFrame([extract_session_text(session_id) for session_id in test_features["session_id"].drop_duplicates().tolist()])\n    model_data = test_features.merge(session_text, on="session_id", how="left", validate="many_to_one")\n    model_data["transcript_text"] = model_data["transcript_text"].fillna("").astype("string")\n    model_data["model_text"] = "[OBJECTIVE] " + model_data["learning_objective"] + "\\n[TRANSCRIPT] " + model_data["transcript_text"]\n\n    matrix_key = model_info["matrix_key"]\n\n    if matrix_key in {"word", "word_char"}:\n        word_vectorizer = joblib.load(ASSET_DIR / "word_tfidf_vectorizer.joblib")\n        word_matrix = word_vectorizer.transform(model_data["model_text"]).tocsr()\n\n    if matrix_key in {"char", "word_char"}:\n        char_vectorizer = joblib.load(ASSET_DIR / "char_tfidf_vectorizer.joblib")\n        char_matrix = char_vectorizer.transform(model_data["model_text"]).tocsr()\n\n    if matrix_key == "word":\n        model_matrix = word_matrix\n    elif matrix_key == "char":\n        model_matrix = char_matrix\n    elif matrix_key == "word_char":\n        model_matrix = sparse.hstack([word_matrix, char_matrix], format="csr")\n    else:\n        raise ValueError(f"Unsupported matrix key: {matrix_key}")\n\n    if model_matrix.shape[1] != len(coef):\n        raise ValueError(f"Feature mismatch: matrix has {model_matrix.shape[1]} columns but model expects {len(coef)}")\n\n    probabilities = stable_sigmoid(np.asarray(model_matrix @ coef).reshape(-1) + intercept)\n    probabilities = np.clip(float(model_info["calibration_alpha"]) * probabilities + (1.0 - float(model_info["calibration_alpha"])) * float(model_info["calibration_prior"]), 1e-6, 1 - 1e-6)\n\n    predictions = pd.DataFrame({"response_id": model_data["response_id"], "prediction": probabilities})\n    submission_format = pd.read_csv(DATA_DIR / "submission_format.csv")\n    submission_format.columns = submission_format.columns.astype(str).str.replace("\\ufeff", "", regex=False).str.strip().str.lower()\n    submission_format["response_id"] = submission_format["response_id"].astype("string").str.strip()\n\n    prediction_columns = [column for column in submission_format.columns if column != "response_id"]\n\n    if len(prediction_columns) != 1:\n        raise ValueError(f"Expected one prediction column, found: {prediction_columns}")\n\n    prediction_column = prediction_columns[0]\n    submission = submission_format[["response_id"]].merge(predictions, on="response_id", how="left", validate="one_to_one").rename(columns={"prediction": prediction_column})\n\n    if submission[prediction_column].isna().any():\n        raise ValueError("Missing predictions found in submission.")\n\n    submission.to_csv(ROOT / "submission.csv", index=False)\n\n\nif __name__ == "__main__":\n    main()\n'

runtime_file = optuna_dir / "main.py"
runtime_file.write_text(runtime_code, encoding="utf-8")
py_compile.compile(str(runtime_file), doraise=True)

print("Runtime syntax check passed.")
print("Runtime file:", runtime_file)

Runtime syntax check passed.
Runtime file: C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Trace-The-Race-Competition\Trace-The-Race-Dataset\outputs\06_model_training\optuna_baseline_v3\main.py


In [19]:
package_dir = submission_dir / "optuna_baseline_package"
asset_dir = package_dir / "assets"

if package_dir.exists():
    shutil.rmtree(package_dir)

asset_dir.mkdir(parents=True, exist_ok=True)

shutil.copy2(runtime_file, package_dir / "main.py")
shutil.copy2(optuna_dir / "linear_model_weights.npz", asset_dir / "linear_model_weights.npz")

if selected_matrix_key in {"word", "word_char"}:
    shutil.copy2(feature_dir / "word_tfidf_vectorizer.joblib", asset_dir / "word_tfidf_vectorizer.joblib")

if selected_matrix_key in {"char", "word_char"}:
    shutil.copy2(feature_dir / "char_tfidf_vectorizer.joblib", asset_dir / "char_tfidf_vectorizer.joblib")

feature_config_file = feature_dir / "feature_config.json"

if feature_config_file.exists():
    shutil.copy2(feature_config_file, asset_dir / "feature_config.json")

model_info = {"model_key": selected_model_key, "matrix_key": selected_matrix_key, "model_family": selected_model_family, "trial_number": selected_trial_number, "parameters": selected_params, "internal_log_loss": float(selected_candidate["internal_log_loss"]), "three_fold_oof_log_loss": float(selected_candidate["cv_log_loss"]), "raw_validation_log_loss": float(raw_validation_loss), "calibrated_validation_log_loss": float(calibrated_validation_loss), "calibration_alpha": float(final_alpha), "calibration_prior": float(final_prior), "full_training_rows": int(len(y_full)), "coefficient_count": int(len(model_coef)), "training_sklearn_version": sklearn.__version__, "optuna_version": optuna.__version__}

with open(asset_dir / "model_info.json", "w", encoding="utf-8") as file:
    json.dump(model_info, file, indent=2)

zip_file = submission_dir / "submission_optuna_baseline.zip"

if zip_file.exists():
    zip_file.unlink()

with zipfile.ZipFile(zip_file, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as archive:
    for file_path in sorted(package_dir.rglob("*")):
        if file_path.is_file():
            archive.write(file_path, file_path.relative_to(package_dir).as_posix())

with zipfile.ZipFile(zip_file, "r") as archive:
    archive_files = archive.namelist()

for required_file in ["main.py", "assets/model_info.json", "assets/linear_model_weights.npz"]:
    if required_file not in archive_files:
        raise ValueError(f"Required file missing from ZIP: {required_file}")

final_summary = pd.DataFrame([{"baseline_public_log_loss": 0.6163, "selected_model": selected_model_key, "selected_matrix": selected_matrix_key, "internal_log_loss": selected_candidate["internal_log_loss"], "three_fold_oof_log_loss": selected_candidate["cv_log_loss"], "raw_validation_log_loss": raw_validation_loss, "calibrated_validation_log_loss": calibrated_validation_loss, "calibration_used": use_calibration, "final_alpha": final_alpha, "zip_file": str(zip_file)}])

final_summary.to_csv(optuna_dir / "final_tuning_summary.csv", index=False)

print("Submission ZIP created successfully.")
print("ZIP file:", zip_file)
print("ZIP size:", f"{zip_file.stat().st_size / 1024**2:.2f} MB")
display(final_summary)

Submission ZIP created successfully.
ZIP file: C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Trace-The-Race-Competition\Trace-The-Race-Dataset\outputs\06_model_training\submissions\submission_optuna_baseline.zip
ZIP size: 1.57 MB


,baseline_public_log_loss,selected_model,selected_matrix,internal_log_loss,three_fold_oof_log_loss,raw_validation_log_loss,calibrated_validation_log_loss,calibration_used,final_alpha,zip_file
0,0.6163,word_char_logistic,word_char,0.570906,0.562109,0.548239,0.548239,True,1.0,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\T...
